In [46]:
from dotenv import load_dotenv
from langgraph.graph import StateGraph,END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage,SystemMessage,AIMessage
from langgraph.checkpoint.memory import MemorySaver
from langchain_groq import ChatGroq
load_dotenv()

True

In [47]:
from typing import TypedDict,Annotated

In [48]:
class AgentState(TypedDict):
    messages : Annotated[list,add_messages]

In [49]:
llm = ChatGroq(model="openai/gpt-oss-120b")

In [50]:
def llm_node(state:AgentState)->dict:
    system = SystemMessage(content="""You are a helpful assistant. 
    Remember everything the user tells you about themselves.""")
    res = llm.invoke([system]+state['messages'])
    return {'messages':[res]}

In [51]:
flow = StateGraph(AgentState)
flow.add_node("node1",llm_node)
flow.set_entry_point("node1")
flow.add_edge("node1",END)

In [52]:
memory = MemorySaver()
app = flow.compile(checkpointer=memory)
# app = flow.compile()

config={"configurable":{"thread_id":"vamsi_session_1"}}

In [53]:
res = app.invoke({'messages':[HumanMessage("Hi My name is Vamsi, I work in TCS")]},
                 config=config)
print(res["messages"][-1].content)

Nice to meet you, Vamsi! It’s great to know you work at TCS. How can I help you today?


In [54]:
res = app.invoke({'messages':[HumanMessage("Hi what is my name")]},
                 config = config)
print(res["messages"][-1].content)

Your name is Vamsi.


In [55]:
res = app.invoke({'messages':[HumanMessage("Where do I work")]},
                 config = config)
print(res["messages"][-1].content)

You work at **TCS** (Tata Consultancy Services).


In [56]:
state_snnipet = app.get_state(config=config)
print(state_snnipet)

StateSnapshot(values={'messages': [HumanMessage(content='Hi My name is Vamsi, I work in TCS', additional_kwargs={}, response_metadata={}, id='a9ab34d0-8b33-4c3a-b0a3-d66b02eee28d'), AIMessage(content='Nice to meet you, Vamsi! It’s great to know you work at TCS. How can I help you today?', additional_kwargs={'reasoning_content': 'We need to respond, acknowledging name and work. Remember for future. Should be friendly.'}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 104, 'total_tokens': 159, 'completion_time': 0.117790936, 'completion_tokens_details': {'reasoning_tokens': 19}, 'prompt_time': 0.014827292, 'prompt_tokens_details': None, 'queue_time': 0.060300158, 'total_time': 0.132618228}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_d81b3304b3', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--019d8a9e-8081-7661-933a-a84e3773e4ea-0', usage_metadata={'input_tokens': 104, 'output_tokens': 55, 'total_tokens':

In [57]:
config_kummu={"configurable":{"thread_id":"kummu_session_1"}}
res = app.invoke({'messages':[HumanMessage("Hi My name is Kummu, I work in TCS-Siruseri")]},
                 config=config_kummu)
print(res["messages"][-1].content)

Hello Kummu! 👋 Nice to meet you. How can I assist you today?


In [58]:
res = app.invoke({'messages':[HumanMessage("Where do I work? and who am I")]},
                 config=config_kummu)
print(res["messages"][-1].content)

You’re **Kummu**, and you work at **TCS‑Siruseri**. Let me know if there’s anything else you’d like to chat about!


In [60]:
snap = app.get_state(config)
print(snap.values)

{'messages': [HumanMessage(content='Hi My name is Vamsi, I work in TCS', additional_kwargs={}, response_metadata={}, id='a9ab34d0-8b33-4c3a-b0a3-d66b02eee28d'), AIMessage(content='Nice to meet you, Vamsi! It’s great to know you work at TCS. How can I help you today?', additional_kwargs={'reasoning_content': 'We need to respond, acknowledging name and work. Remember for future. Should be friendly.'}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 104, 'total_tokens': 159, 'completion_time': 0.117790936, 'completion_tokens_details': {'reasoning_tokens': 19}, 'prompt_time': 0.014827292, 'prompt_tokens_details': None, 'queue_time': 0.060300158, 'total_time': 0.132618228}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_d81b3304b3', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--019d8a9e-8081-7661-933a-a84e3773e4ea-0', usage_metadata={'input_tokens': 104, 'output_tokens': 55, 'total_tokens': 159}), HumanMessage(

In [62]:
for checkpoint in app.get_state_history(config):
    print(f"Step: {checkpoint.metadata['step']}")
    print(f"Messages: {checkpoint.values['messages']}")
    print(f"Checkpoint ID: {checkpoint.config['configurable']['checkpoint_id']}")
    print("---")

Step: 7
Messages: [HumanMessage(content='Hi My name is Vamsi, I work in TCS', additional_kwargs={}, response_metadata={}, id='a9ab34d0-8b33-4c3a-b0a3-d66b02eee28d'), AIMessage(content='Nice to meet you, Vamsi! It’s great to know you work at TCS. How can I help you today?', additional_kwargs={'reasoning_content': 'We need to respond, acknowledging name and work. Remember for future. Should be friendly.'}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 104, 'total_tokens': 159, 'completion_time': 0.117790936, 'completion_tokens_details': {'reasoning_tokens': 19}, 'prompt_time': 0.014827292, 'prompt_tokens_details': None, 'queue_time': 0.060300158, 'total_time': 0.132618228}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_d81b3304b3', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--019d8a9e-8081-7661-933a-a84e3773e4ea-0', usage_metadata={'input_tokens': 104, 'output_tokens': 55, 'total_tokens': 159}), HumanMes